# wrangle / portfolio / return

**Intent**
- Gives low, base and high scenarios as compound rates in percent per year, as constants, and must:
  - Give `portfolio_return_nominal_{low,base,high}_pct`, the nominal annual return to 2070.
  - Grow the portfolio in the plan.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is ten years of annual portfolio returns.
- It is synthetic until the `wrangle` series is loaded.
- The `equity` table holds prices and changes, but portfolio weights are only in the Portfolio sheet.

**Status**
- The base is a placeholder, the mean annual return.
- Low and high are one standard deviation either side.
- It carries no evaluation, so the golden replay is the gate.

**Future**
- Overall: the work still to do before the first release:
  - Promote the target from local to served, then add the Register stage, once the limits are fixed against real data.
  - Add `query_postgres` to `collect_run` and load the real series in place of the synthetic ones.
  - Land the portfolio weights in Postgres, and use total return, not price return.
  - Blend the history with forward capital market assumptions, since 15 years cannot tell 6 percent from 7.
  - Bootstrap monthly returns in blocks to 2070, re-centred on the forward base, and take the compound rate.
  - Set base as the median and low and high as the 25th and 75th percentiles.
  - Give the real return by subtracting the matching `household/costs` growth, and allow for fees and tax.
  - Serve the numbers to the planning sheet through `wrangle` and an Apps Script function, which is not yet built.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/portfolio/return")

ws.inputs[0] = ws.inputs[0]

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

DEVIATIONS = [-4.0, -3.0, -2.0, -1.0, 0.0, 0.0, 1.0, 2.0, 3.0, 4.0]
collect_run(
    ws,
    1000,
    label="Ten years of synthetic annual portfolio returns",
    name="portfolio_return_training",
    source="builder",
    build=lambda: {"year": list(range(2016, 2026)), "portfolio_return_pct": [round(6.0 + deviation, 4) for deviation in DEVIATIONS]},
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "portfolio_return_training"))
print("training", ws.training.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
BASE = round(float(ws.training["portfolio_return_pct"].mean()), 2)
SPREAD = round(float(ws.training["portfolio_return_pct"].std(ddof=0)), 2)
ws.params = dict(zip(ws.outputs, [round(BASE - SPREAD, 2), BASE, round(BASE + SPREAD, 2)], strict=True))
print("constants", ws.params)
assert list(ws.params.values()) == [3.55, 6.0, 8.45], "the fit did not recover the mean and spread"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

define_model(ws, model=1000, flavour="static", source=ws.params, label="Mean annual portfolio return and one standard deviation either side")

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [1.0, None], "expect": ["ok", "error"]}
package_model(ws, 1000, golden, target={"kind": "local"})